In [1]:
import pandas as pd

In [2]:
from Descriptive import Descriptive

In [3]:
obj=Descriptive()

In [4]:
from nsepy import get_history as gh
import datetime as dt
import yfinance as yf
stock_symbol = "RELIANCE.NS" #NSE stocks usually end with .NS
#dowload the stock data from NSE
stk_data=yf.download(stock_symbol, start="2024-05-01", end="2025-10-09")

[*********************100%***********************]  1 of 1 completed


In [5]:
stk_data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 358 entries, 2024-05-02 to 2025-10-08
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   (Close, RELIANCE.NS)   358 non-null    float64
 1   (High, RELIANCE.NS)    358 non-null    float64
 2   (Low, RELIANCE.NS)     358 non-null    float64
 3   (Open, RELIANCE.NS)    358 non-null    float64
 4   (Volume, RELIANCE.NS)  358 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 16.8 KB


In [6]:
print(stk_data.columns)

MultiIndex([( 'Close', 'RELIANCE.NS'),
            (  'High', 'RELIANCE.NS'),
            (   'Low', 'RELIANCE.NS'),
            (  'Open', 'RELIANCE.NS'),
            ('Volume', 'RELIANCE.NS')],
           names=['Price', 'Ticker'])


In [7]:
exog = stk_data[[('Volume', 'RELIANCE.NS')]]

In [8]:
stk_data=stk_data[["Open","High","Low","Close"]]

In [9]:
stk_data

Price,Open,High,Low,Close
Ticker,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS
Date,,,,
2024-05-02,1454.460329,1459.721831,1446.679164,1449.075317
2024-05-03,1453.472085,1457.374970,1399.275682,1416.912964
2024-05-06,1418.395221,1422.841601,1401.103743,1402.610596
2024-05-07,1399.102859,1403.820987,1375.413559,1384.775635
2024-05-08,1380.848112,1415.875659,1380.848112,1401.647339
...,...,...,...,...
2025-10-01,1360.708629,1372.255218,1356.428371,1362.400757
2025-10-03,1356.926092,1365.287457,1350.655159,1357.125244


In [10]:
from sklearn.preprocessing import MinMaxScaler
scaler1 = MinMaxScaler()
data1 = pd.DataFrame(scaler1.fit_transform(stk_data), columns=['Open','High','Low','Close'])
scaler2 = MinMaxScaler()
exog = pd.DataFrame(scaler2.fit_transform(exog), columns=['Volume'])

In [11]:
#DataFrame containing stock prices and a list of column names to use
data1=pd.DataFrame(data1,columns=["Open","High","Low","Close"])

In [12]:
#Training and test set split
training_size = round(len(data1 ) * 0.80)
print(training_size)
X_train=data1[:training_size]
X_test=data1[training_size:]
print("X_train length:",X_train.shape)
print("X_test length:",X_test.shape)
y_train=data1[:training_size]
y_test=data1[training_size:]
print("y_train length:",y_train.shape)
print("y_test length:",y_test.shape)

286
X_train length: (286, 4)
X_test length: (72, 4)
y_train length: (286, 4)
y_test length: (72, 4)


In [13]:
from sklearn.metrics import mean_squared_error
#Imports the function to calculate Mean Squared Error (MSE)
from sklearn.metrics import mean_absolute_percentage_error
#Imports the function to calculate MAPE (Mean Absolute Percentage Error)
import numpy as np

In [14]:
from statsmodels.tsa.holtwinters import SimpleExpSmoothing
#Imports the SES (Simple Exponential Smoothing) to create a model multiple variables together

In [20]:
def combination_SES(dataset, listt):
# Defines a function for Simple Exponential Smoothing
    performance = {"Model": [], "RMSE": [], "MaPe": [], "Lag": [], "Test": []}
    print(listt)
    # Select required columns
    datasetTwo = dataset[listt]
    test_obs = 28
    # Last 28 observations are used as test data
    train = datasetTwo[:-test_obs]
    # Training dataset
    test = datasetTwo[-test_obs:]
    # Testing dataset

    from statsmodels.tsa.holtwinters import SimpleExpSmoothing
    from sklearn.metrics import mean_squared_error
    from sklearn.metrics import mean_absolute_percentage_error

    forecasts = pd.DataFrame(index=test.index, columns=listt)
    # Fit SES model for each variable
    for column in listt:
        print("\nModel:", column)
        # Create SES model
        model = SimpleExpSmoothing(train[column],initialization_method="estimated")
        # Fit model
        result = model.fit(optimized=True)
        # Optimized alpha value
        print("Alpha:", result.params["smoothing_level"])
        # Forecast next 28 observations
        pred = result.forecast(test_obs)
        forecasts[column] = pred
    # Save forecasts
    forecasts.to_csv(f"SES_forecasted_{test_obs}.csv")
    # Calculate performance
    rmse = np.sqrt(mean_squared_error(test, forecasts))
    mape = mean_absolute_percentage_error(test,forecasts)

    performance["Model"].append(listt)
    performance["RMSE"].append(rmse)
    performance["MaPe"].append(mape)
    # SES has no lag order like VARMA
    performance["Lag"].append("SES")
    performance["Test"].append(test_obs)
    perf = pd.DataFrame(performance)
    return perf, result, forecasts

In [21]:
listt=["Close","High","Open","Low"]

In [23]:
perf, result, pred = combination_SES(data1, listt)

['Close', 'High', 'Open', 'Low']

Model: Close
Alpha: 0.9777825473046834

Model: High
Alpha: 0.9999999850988388

Model: Open
Alpha: 0.9999999850988388

Model: Low
Alpha: 0.9999999850988388


In [ ]:
data1

In [ ]:
perf